# Aşama 1A – NEU-DET Veri İnceleme

Bu notebook, **NEU-DET** veri setinin gerçek içeriğini inceler ve raporlar.

## Ne yapar?
1. Colab ortamını ve GPU'yu kontrol eder (GPU zorunlu değil).
2. GitHub'dan bu projenin güncel kodunu çeker.
3. Google Drive'ı bağlar ve veri seti ZIP dosyasını bulur.
4. ZIP'i `/content/neu_det_raw/` klasörüne çıkarır.
5. Görüntü ↔ XML eşleşmesini, sınıf dağılımını ve veri sorunlarını raporlar.
6. Üç çıktı üretir: `data_summary.json`, `class_distribution.png`, `annotation_samples.png`.
7. Bu çıktıları hem repo içindeki `results/` klasörüne hem Google Drive `outputs/` klasörüne kaydeder.

## Ne yapmaz? (Aşama 1A kapsamı dışı)
- Train / Validation / Test ayrımı yapmaz.
- XML → YOLO dönüşümü yapmaz.
- Model eğitmez, tahmin yapmaz.
- Streamlit uygulaması oluşturmaz.

> **Not:** Hücreleri yukarıdan aşağıya sırayla çalıştırın. Kritik bir hücre hata verirse
> sonraki hücrelere geçmeyin; hata mesajı sorunun ne olduğunu söyler.

## A) Ortam ve GPU kontrolü

Önce Python ve PyTorch sürümünü, ardından GPU durumunu yazdırıyoruz.

**Neden gerekli?** Eğitim aşamasında GPU şart, ancak bu aşamada sadece veri okunuyor.
GPU yoksa bilgi verip devam ediyoruz.

In [ ]:
import sys

import torch  # Colab'da PyTorch kurulu gelir.

print("Python sürümü :", sys.version.split()[0])
print("PyTorch sürümü:", torch.__version__)
print("CUDA derlemesi:", torch.version.cuda)

cuda_available = torch.cuda.is_available()
print("CUDA kullanılabilir mi:", cuda_available)

if cuda_available:
    device_index = torch.cuda.current_device()
    properties = torch.cuda.get_device_properties(device_index)
    print("GPU modeli    :", torch.cuda.get_device_name(device_index))
    print("VRAM          : %.2f GB" % (properties.total_memory / (1024 ** 3)))
else:
    print()
    print("UYARI: GPU bulunamadı.")
    print("Bu veri inceleme aşaması CPU üzerinde de çalışır, işlem devam edebilir.")
    print("Ancak Aşama 2'deki model eğitimi için Colab'da GPU seçili olmalıdır.")
    print("Colab menüsü: Runtime -> Change runtime type -> Hardware accelerator -> GPU")

## Gerekli kütüphanelerin kontrolü

Bu aşama için standart kütüphaneler + **Pillow** (görüntü okuma) + **Matplotlib** (grafik) yeterli.

Aşağıdaki hücre sadece **eksik** paketi kurar; toplu kurulum veya sürüm yükseltmesi yapmaz.
`requirements.txt` bu aşamada değiştirilmiyor.

In [ ]:
import importlib.util

required_packages = {"PIL": "Pillow", "matplotlib": "matplotlib", "numpy": "numpy"}

missing = []
for module_name, package_name in required_packages.items():
    installed = importlib.util.find_spec(module_name) is not None
    print(f"{package_name:12s} kurulu mu: {installed}")
    if not installed:
        missing.append(package_name)

if missing:
    print("Eksik paketler:", missing)
    print("Sadece eksik olanlar kuruluyor...")
    import sys as _sys
    import subprocess as _subprocess

    _subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Gerekli paketlerin hepsi mevcut, kurulum yapılmadı.")

## B) GitHub repository bağlantısı

Projenin güncel kodu `/content/celik-kusur-tespiti` klasörüne alınır.

**Neden?** Kod GitHub'da tutuluyor; Colab her açıldığında boş bir makine gibi başlar.
Bu yüzden kodu klonluyoruz (zaten varsa sadece güncelliyoruz) ve `src` klasörünü
Python'un arama yoluna ekliyoruz.

Bu hücre hata verirse **durmayın, aşağı inin**; hata mesajı neyin yanlış olduğunu söyler.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/mertkk1623/celik-kusur-tespiti.git"
REPO_DIR = Path("/content/celik-kusur-tespiti")

if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(
        f"{REPO_DIR} klasörü var ama bir git deposu değil. "
        "Bu klasörü silip hücreyi yeniden çalıştırın."
    )

if (REPO_DIR / ".git").is_dir():
    # Klasör zaten klonlanmış: yeniden klonlamıyoruz, sadece güncel kodu çekiyoruz.
    print("Repo zaten mevcut, yeniden klonlanmıyor:", REPO_DIR)
    pull = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
    print(pull.stdout.strip() or pull.stderr.strip())
    if pull.returncode != 0:
        print("UYARI: 'git pull' başarısız oldu. Mevcut kod kullanılacak.")
        print("Sebep:", pull.stderr.strip())
else:
    print("Repo klonlanıyor:", REPO_URL)
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True,
        text=True,
    )
    if clone.returncode != 0:
        raise RuntimeError(
            "Repository klonlanamadı.\n"
            f"Komut çıktısı:\n{clone.stderr.strip()}\n"
            "İnternet bağlantısını ve repo adresini kontrol edin."
        )
    print("Klonlama tamam:", REPO_DIR)

# src klasörünü Python'un import yoluna ekliyoruz; böylece 'import data_inspection' çalışır.
SRC_DIR = REPO_DIR / "src"
if not SRC_DIR.is_dir():
    raise RuntimeError(f"src klasörü bulunamadı: {SRC_DIR}")
if not (SRC_DIR / "data_inspection.py").is_file():
    raise RuntimeError(
        f"data_inspection.py bulunamadı: {SRC_DIR / 'data_inspection.py'}\n"
        "Dosyalar GitHub'a push edilmiş mi kontrol edin."
    )
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import data_inspection
except Exception as error:
    raise RuntimeError(f"data_inspection modülü import edilemedi: {error}") from error

# Colab'da kod güncellenmiş olabilir; modülü tazelemek eski kodu kullanmayı önler.
importlib.reload(data_inspection)

print("Modül hazır   :", data_inspection.__file__)
print("Beklenen sınıflar:", ", ".join(data_inspection.EXPECTED_CLASSES))

## C) Google Drive bağlantısı

Veri seti ZIP dosyası ve çıktılar Drive'da. Bağlantı kurulurken Google hesabı
yetkilendirmesi istenir; tarayıcıda hesabınızı seçip izin verin.

**Neden?** Colab oturumu kapandığında `/content` silinir, Drive kalıcıdır.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.is_dir():
    raise RuntimeError(
        f"Google Drive bağlanamadı: {DRIVE_ROOT} erişilebilir değil.\n"
        "Drive izinlerini verdiğinizden emin olun ve önceki hücreyi tekrar çalıştırın."
    )

DRIVE_READY = True
print("Drive bağlantısı doğrulandı:", DRIVE_ROOT)

## 4) Veri seti ZIP dosyasını bulma

ZIP dosyasının yeri kesin bilinmediği için önce varsayılan yol denenir, sonra
MyDrive içinde ismi `NEU-DET`, `NEU_DET` veya `NEUDET` olan ZIP dosyaları aranır.

Kurallar:
- Tek aday bulunursa otomatik seçilir.
- Birden fazla aday bulunursa liste gösterilir ve sizden seçim istenir.
- Hiç aday yoksa işlem durur ve `DATA_ZIP_PATH` değerini elle girmeniz istenir.

> **Yolu elle vermek için:** aşağıdaki hücredeki `DATA_ZIP_PATH` satırını
> kendi dosya yolunuzla değiştirip hücreyi yeniden çalıştırın.

In [ ]:
import json
import zipfile
from pathlib import Path

# ---------------------------------------------------------------------------
# AYAR: ZIP dosyasının yolu. Biliniyorsa buraya tam yol yazılabilir, örneğin:
# DATA_ZIP_PATH = "/content/drive/MyDrive/celik-kusur-tespiti/dataset/NEU-DET.zip"
# Bilinmiyorsa None bırakın; aşağıdaki arama devreye girer.
# ---------------------------------------------------------------------------
DATA_ZIP_PATH = "/content/drive/MyDrive/celik-kusur-tespiti/dataset/NEU-DET.zip"

EXTRACT_DIR = Path("/content/neu_det_raw")


def find_dataset_zips(search_root):
    """Drive altında adı NEU-DET / NEU_DET / NEUDET olan ZIP dosyalarını bulur.

    Karşılaştırma için isimdeki boşluk, tire ve alt çizgi kaldırılır;
    böylece 'NEU-DET.zip', 'NEU_DET.zip' ve 'neudet.zip' aynı kabul edilir.
    """
    search_root = Path(search_root)
    candidates = []
    for path in search_root.rglob("*"):
        if not path.is_file() or path.suffix.lower() != ".zip":
            continue
        normalized = path.stem.lower().replace("-", "").replace("_", "").replace(" ", "")
        if "neudet" in normalized:
            candidates.append(path)
    return sorted(candidates, key=lambda p: str(p).lower())


def resolve_zip_path(configured_path, search_root):
    """Kullanılacak ZIP yolunu belirler; belirsizlik varsa kullanıcıya sorar."""
    if configured_path:
        candidate = Path(configured_path)
        if candidate.is_file():
            print("ZIP bulundu (ayarlanan yol):", candidate)
            return candidate
        print("Ayarlanan yolda dosya yok, arama yapılacak:", candidate)

    candidates = find_dataset_zips(search_root)
    if not candidates:
        raise RuntimeError(
            "NEU-DET ZIP dosyası bulunamadı.\n"
            "Lütfen ZIP dosyasını Drive'a yükleyin ve bu hücredeki DATA_ZIP_PATH "
            "değişkenine tam yolu yazın.\n"
            "Örnek: DATA_ZIP_PATH = '/content/drive/MyDrive/.../NEU-DET.zip'"
        )

    if len(candidates) == 1:
        print("Tek aday bulundu, otomatik seçildi:", candidates[0])
        return candidates[0]

    print("Birden fazla ZIP dosyası bulundu:")
    for index, path in enumerate(candidates, start=1):
        print(f"  [{index}] {path}")
    answer = input("Kullanmak istediğiniz dosyanın numarasını girin: ").strip()
    if not answer.isdigit() or not (1 <= int(answer) <= len(candidates)):
        raise RuntimeError("Geçersiz seçim yapıldı. Hücreyi tekrar çalıştırın.")
    selected = candidates[int(answer) - 1]
    print("Seçilen dosya:", selected)
    return selected


ZIP_PATH = resolve_zip_path(DATA_ZIP_PATH, DRIVE_ROOT)

if not ZIP_PATH.is_file():
    raise RuntimeError(f"ZIP dosyası gerçekten mevcut değil: {ZIP_PATH}")

print("ZIP boyutu: %.1f MB" % (ZIP_PATH.stat().st_size / (1024 ** 2)))

## ZIP dosyasını doğrulama ve çıkarma

Zip bozuksa burada açık hata verilir. Çıkarma hedefi `/content/neu_det_raw/`.

**Tekrar çalıştırma güvenliği:** Hedef klasörde bir `.extract_info.json` işaret dosyası tutulur.
Aynı ZIP daha önce çıkarıldıysa tekrar çıkarılmaz (gereksiz iş ve zaman kaybı olmaz).
Klasördeki başka dosyalar **silinmez**; sadece aynı isimli dosyalar üzerine yazılır.

In [ ]:
import json
import zipfile
from pathlib import Path

MARKER_FILE = EXTRACT_DIR / ".extract_info.json"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


def validate_zip(zip_path):
    """ZIP dosyasının açılabilir olduğunu kontrol eder; bozuksa hata verir."""
    if not zip_path.is_file():
        raise FileNotFoundError(f"ZIP dosyası bulunamadı: {zip_path}")
    try:
        with zipfile.ZipFile(zip_path) as archive:
            corrupt_member = archive.testzip()
            if corrupt_member is not None:
                raise RuntimeError(
                    f"ZIP bozuk: '{corrupt_member}' dosyası okunamıyor ({zip_path})."
                )
            member_count = len(archive.infolist())
    except zipfile.BadZipFile as error:
        raise RuntimeError(f"ZIP dosyası açılamıyor (bozuk olabilir): {zip_path} -> {error}") from error
    return member_count


def extract_zip_if_needed(zip_path, target_dir, marker_file):
    """ZIP'i hedef klasöre çıkarır; aynı ZIP daha önce çıkarıldıysa atlar."""
    zip_stat = zip_path.stat()
    current_info = {
        "zip_path": str(zip_path.resolve()),
        "zip_size": zip_stat.st_size,
        "zip_mtime": int(zip_stat.st_mtime),
    }

    # Klasörde işaret dosyası dışında gerçek içerik var mı?
    # (İşaret dosyası tek başınaysa veri yok sayılır ve yeniden çıkarılır.)
    has_content = any(item.name != marker_file.name for item in target_dir.iterdir())

    if marker_file.is_file() and has_content:
        try:
            previous_info = json.loads(marker_file.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            previous_info = None
        if previous_info == current_info:
            print("Bu ZIP daha önce çıkarılmış, tekrar çıkarılmıyor:", target_dir)
            return False

    print("ZIP çıkarılıyor ->", target_dir)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            # macOS'un eklediği gereksiz klasörü atlıyoruz.
            if "__MACOSX" in member.filename:
                continue
            archive.extract(member, target_dir)

    marker_file.write_text(json.dumps(current_info, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Çıkarma tamamlandı.")
    return True


member_count = validate_zip(ZIP_PATH)
print("ZIP doğrulandı. İçindeki öğe sayısı:", member_count)

extracted_now = extract_zip_if_needed(ZIP_PATH, EXTRACT_DIR, MARKER_FILE)
print("Bu çalıştırmada çıkarma yapıldı mı:", extracted_now)

## 5–6) Veri seti incelemesi

`discover_dataset_pairs`, `parse_voc_annotation` ve `inspect_dataset` fonksiyonları
`src/data_inspection.py` içinde tanımlı. Bu hücre onları çağırıp sonucu yazdırır.

**Önemli ayrım:**
- **Görüntü sayısı**: o sınıfı içeren görüntü dosyası sayısı.
- **Bounding box sayısı**: o sınıfa ait kutu sayısı.

Bir görüntüde aynı sınıftan birden fazla kusur olabilir; bu yüzden iki sayı farklıdır.

In [ ]:
import json
from pathlib import Path

import data_inspection

summary = data_inspection.inspect_dataset(EXTRACT_DIR)
discovered = data_inspection.discover_dataset_pairs(EXTRACT_DIR)

print("=" * 62)
print("TEMEL SAYILAR")
print("=" * 62)
print("Toplam görüntü sayısı       :", summary["total_images"])
print("Toplam XML sayısı           :", summary["total_xml_annotations"])
print("Eşleşen görüntü/XML çifti   :", summary["matched_pairs"])
print("Eksik XML (görüntü var)     :", summary["missing_xml_count"])
print("Eşleşmeyen XML (görüntü yok):", summary["unmatched_xml_count"])
print("Hatalı XML sayısı           :", summary["invalid_annotation_count"])
print("Okunamayan görüntü sayısı   :", summary["unreadable_image_count"])
print("Kutusu olmayan görüntü      :", summary["images_without_objects_count"])
print("Birden fazla sınıf içeren   :", summary["images_with_multiple_classes"])

print()
print("=" * 62)
print("SINIF DAĞILIMI")
print("=" * 62)
print(f"{'Sınıf':<18}{'Görüntü':>10}{'Bounding box':>15}")
print("-" * 62)
for class_name in data_inspection.EXPECTED_CLASSES:
    print(
        f"{class_name:<18}"
        f"{summary['class_image_counts'].get(class_name, 0):>10}"
        f"{summary['class_bbox_counts'].get(class_name, 0):>15}"
    )

print()
print("=" * 62)
print("GÖRÜNTÜ BOYUTLARI")
print("=" * 62)
dimensions = summary["image_dimensions"]
print("Beklenen boyut           :", dimensions["expected"])
print("En sık görülen boyut     :", dimensions["most_common"])
print("Benzersiz boyut dağılımı :", dimensions["unique_size_counts"])
print("Genişlik aralığı         :", dimensions["min_width"], "-", dimensions["max_width"])
print("Yükseklik aralığı        :", dimensions["min_height"], "-", dimensions["max_height"])
print("Beklenen dışı görüntü    :", dimensions["non_standard_size_count"])

print()
print("=" * 62)
print("VERİ SORUNLARI")
print("=" * 62)
print("Beklenmeyen sınıf adları :", summary["unexpected_class_names"] or "yok")

if summary["invalid_annotations"]:
    print()
    print("Hatalı XML dosyaları (ilk 10):")
    for item in summary["invalid_annotations"][:10]:
        print(" -", item["file"])
        for message in item["errors"]:
            print("     *", message)
    if len(summary["invalid_annotations"]) > 10:
        print(" ... toplam", len(summary["invalid_annotations"]), "hatalı XML var.")

if summary["missing_xml_files"]:
    print()
    print("XML'i olmayan görüntüler (ilk 10):")
    for path in summary["missing_xml_files"][:10]:
        print(" -", path)

if summary["unmatched_xml_files"]:
    print()
    print("Görüntüsü olmayan XML dosyaları (ilk 10):")
    for path in summary["unmatched_xml_files"][:10]:
        print(" -", path)

if summary["images_without_objects"]:
    print()
    print("Geçerli ama hiç kutusu olmayan XML'ler (ilk 10):")
    for path in summary["images_without_objects"][:10]:
        print(" -", path)

if summary["unreadable_images"]:
    print()
    print("Okunamayan görüntüler (ilk 10):")
    for item in summary["unreadable_images"][:10]:
        print(" -", item["file"], "->", item["error"])

## 7) Görsel çıktılar

İki grafik üretilir:

1. `class_distribution.png` – altı sınıfın **görüntü sayısı** dağılımı.
2. `annotation_samples.png` – 6–8 gerçek görüntü üzerine XML kutuları ve sınıf adları.

Kutular XML'deki piksel koordinatlarıyla çizilir; görüntüler kırpılmaz veya büyütülmez.
Örnek seçimi `seed=42` ile tekrarlanabilir.

In [ ]:
import json
from datetime import datetime, timezone

RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# JSON raporu: istenen tüm alanlar burada.
# ---------------------------------------------------------------------------
report = {
    "stage": "1A",
    "seed": 42,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "dataset_root": summary["dataset_root"],
    "total_images": summary["total_images"],
    "total_xml_annotations": summary["total_xml_annotations"],
    "matched_pairs": summary["matched_pairs"],
    "class_image_counts": summary["class_image_counts"],
    "class_bbox_counts": summary["class_bbox_counts"],
    "image_dimensions": summary["image_dimensions"],
    "missing_xml_count": summary["missing_xml_count"],
    "missing_xml_files": summary["missing_xml_files"],
    "unmatched_image_count": summary["unmatched_image_count"],
    "unmatched_image_files": summary["unmatched_image_files"],
    "unmatched_xml_count": summary["unmatched_xml_count"],
    "unmatched_xml_files": summary["unmatched_xml_files"],
    "invalid_annotation_count": summary["invalid_annotation_count"],
    "invalid_annotations": summary["invalid_annotations"],
    "unreadable_image_count": summary["unreadable_image_count"],
    "unreadable_images": summary["unreadable_images"],
    "images_without_objects_count": summary["images_without_objects_count"],
    "images_without_objects": summary["images_without_objects"],
    "images_with_multiple_classes": summary["images_with_multiple_classes"],
    "unexpected_class_names": summary["unexpected_class_names"],
}

summary_json_path = RESULTS_DIR / "data_summary.json"
summary_json_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("JSON yazıldı:", summary_json_path)

# ---------------------------------------------------------------------------
# Grafikler
# ---------------------------------------------------------------------------
distribution_path = data_inspection.plot_class_distribution(
    summary,
    RESULTS_DIR / "class_distribution.png",
)
print("Sınıf dağılımı grafiği:", distribution_path)

samples_path = data_inspection.plot_annotation_samples(
    discovered["matched_pairs"],
    RESULTS_DIR / "annotation_samples.png",
    max_samples=8,
    seed=42,
)
print("Örnek anotasyon görseli:", samples_path)

GENERATED_FILES = [summary_json_path, distribution_path, samples_path]
print()
print("Üretilen dosyalar:")
for path in GENERATED_FILES:
    print(f" - {path}  ({path.stat().st_size / 1024:.1f} KB)")

## 8) Google Drive'a kaydetme

Üretilen üç dosya `/content/drive/MyDrive/celik-kusur-tespiti/outputs/` klasörüne
kopyalanır ve gerçekten kaydedildiği kontrol edilir.

Drive bağlantısı kurulmadıysa bu hücre hata verir; kayıt başarılı gibi gösterilmez.

In [ ]:
import shutil
from pathlib import Path

if not globals().get("DRIVE_READY", False):
    raise RuntimeError(
        "Google Drive bağlantısı doğrulanmadı. "
        "C bölümündeki mount hücresini çalıştırıp bu hücreyi tekrar deneyin."
    )

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/celik-kusur-tespiti/outputs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Drive hedef klasörü:", DRIVE_OUTPUT_DIR)
print()

saved_paths = []
for source_path in GENERATED_FILES:
    source_path = Path(source_path)
    if not source_path.is_file():
        raise FileNotFoundError(f"Kaynak dosya bulunamadı: {source_path}")

    destination = DRIVE_OUTPUT_DIR / source_path.name
    shutil.copy2(source_path, destination)

    if not destination.is_file() or destination.stat().st_size == 0:
        raise RuntimeError(f"Dosya Drive'a kaydedilemedi: {destination}")

    saved_paths.append(destination)
    print(f"Kaydedildi: {destination}  ({destination.stat().st_size / 1024:.1f} KB)")

print()
print("Tüm dosyalar hem repo hem Drive'da:")
for repo_path, drive_path in zip(GENERATED_FILES, saved_paths):
    print(f" - repo : {repo_path}")
    print(f"   drive: {drive_path}")

## 13) Sonuç özeti ve sıradaki adım

Aşağıdaki hücre, raporun en önemli sayılarını tek ekranda toplar.

**Sonraki adım:** Bu çıktılar (özellikle `results/data_summary.json`) incelenip
veri setinde sorun olmadığı doğrulandıktan sonra Aşama 1B'ye (split + YOLO dönüşümü)
geçilecek. Bu notebook split oluşturmaz ve dönüşüm yapmaz.

In [ ]:
print("=" * 62)
print("AŞAMA 1A ÖZETİ")
print("=" * 62)
print("Toplam görüntü       :", report["total_images"])
print("Toplam XML           :", report["total_xml_annotations"])
print("Toplam bounding box  :", sum(report["class_bbox_counts"].values()))
print("Hatalı XML           :", report["invalid_annotation_count"])
print("Eksik XML            :", report["missing_xml_count"])
print("Eşleşmeyen XML       :", report["unmatched_xml_count"])
print("Beklenmeyen sınıf    :", report["unexpected_class_names"] or "yok")
print()
print("Sınıf başına görüntü / kutu:")
for class_name in data_inspection.EXPECTED_CLASSES:
    images_count = report["class_image_counts"].get(class_name, 0)
    bbox_count = report["class_bbox_counts"].get(class_name, 0)
    print(f"  {class_name:<18} {images_count:>5} görüntü / {bbox_count:>5} kutu")
print()
print("Repo çıktıları  :", RESULTS_DIR)
print("Drive çıktıları :", DRIVE_OUTPUT_DIR)
print()
print("Bu çıktılar Aşama 1B'ye girdi olarak kullanılacak.")

---
# Aşama 1B – Veri Ayrımı (Train / Validation / Test)

Bu bölüm NEU-DET veri setini **görüntü bazında** üç gruba ayırır:

| Grup | Oran | Beklenen görüntü |
|---|---|---|
| Train | %70 | 1260 |
| Validation | %15 | 270 |
| Test | %15 | 270 |

Rastgelelik tohumu: **42**

## Neden görüntü bazında?
Aşama 1A incelemesinde bazı görüntülerde birden fazla kusur sınıfı vardı.
Bir görüntünün bir kutusunu train'e, diğerini test'e koymak **veri sızıntısı**
olur ve ölçülen başarımı yanıltır. Bu yüzden görüntünün **tüm anotasyonları
tek bir grupta** kalır; her görüntü CSV'de tek satırdır.

## Not: sınıf sayıları toplamı görüntü sayısından büyük olabilir
`defect_classes` alanı görüntüdeki bütün sınıfları taşır. Bir görüntü iki sınıf
içeriyorsa hem "crazing" hem "patches" sayımına girer. Bu yüzden aşağıdaki
sınıf dağılımı tablolarında sütun toplamları, o split'in görüntü sayısından
**fazla** çıkabilir. Bu bir hata değil, çok etiketli verinin doğal sonucudur.

> **ÖNEMLİ – Test seti kuralı:** Test seti **yalnızca** nihai başarım ölçümü
> için ayrılmıştır. Model eğitiminde, hiperparametre seçiminde, erken durdurma
> kararında veya herhangi bir ayar/optimizasyon döngüsünde **kullanılmayacaktır**.
> Geliştirme sırasında tüm kararlar train + validation üzerinden verilir.

## B1) Modülü hazırlama

`src/split_dataset.py` dosyası Aşama 1A'da kullandığımız `data_inspection.py`
fonksiyonlarını (görüntü/XML eşleştirme ve XML okuma) yeniden kullanır;
aynı XML okuma kodu ikinci kez yazılmamıştır.

Repo bu hücreden önce güncellenmiş olmalı. Kod güncel değilse hücre hata verir.

In [ ]:
import importlib
import sys
from pathlib import Path

# Aşama 1A hücresinde SRC_DIR ve REPO_DIR tanımlanmıştı; yine de garantiye alıyoruz.
if "REPO_DIR" not in globals():
    raise RuntimeError(
        "REPO_DIR tanımlı değil. Lütfen önce Aşama 1A hücrelerini (B bölümü) çalıştırın."
    )

SRC_DIR = Path(REPO_DIR) / "src"
if not (SRC_DIR / "split_dataset.py").is_file():
    raise RuntimeError(
        f"split_dataset.py bulunamadı: {SRC_DIR / 'split_dataset.py'}\n"
        "Dosyalar GitHub'a push edilmiş mi kontrol edin ve repo'yu güncelleyin."
    )
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import split_dataset
except Exception as error:
    raise RuntimeError(f"split_dataset modülü import edilemedi: {error}") from error

importlib.reload(split_dataset)

print("Modül hazır :", split_dataset.__file__)
print("Seed        :", split_dataset.DEFAULT_SEED)
print("Hedef sayılar:", split_dataset.TARGET_COUNTS)

## B2) Ayrımı oluşturma ve CSV yazma

Bu hücre her görüntü için tek kayıt oluşturur, seed 42 ile karıştırır ve
`splits/neu_det_split.csv` dosyasını yazar.

CSV sütunları: `image_rel_path`, `annotation_rel_path`, `defect_classes`, `split`

Yollar `/content/neu_det_raw` köküne göre görelidir; böylece dosya başka bir
makinede de okunabilir.

In [ ]:
from pathlib import Path

SPLIT_CSV_PATH = Path(REPO_DIR) / "splits" / "neu_det_split.csv"

split_result = split_dataset.create_split(
    dataset_root=EXTRACT_DIR,
    output_csv=SPLIT_CSV_PATH,
    seed=42,
)

print("CSV yazıldı          :", split_result["csv_path"])
print("Toplam görüntü       :", split_result["total_images"])
print("Hedef 1800 mü        :", split_result["total_images_matches_target"])
print("Çok sınıflı görüntü  :", split_result["multi_class_image_count"])

if split_result["warnings"]:
    print()
    print("UYARILAR (ilk 10):")
    for item in split_result["warnings"][:10]:
        print(" -", item["file"])
        print("     ", item["message"])
    if len(split_result["warnings"]) > 10:
        print(" ... toplam", len(split_result["warnings"]), "uyarı var.")
else:
    print("Uyarı yok: tüm XML'ler okunabildi ve her görüntüde en az bir sınıf var.")

## B3) Train / Validation / Test görüntü sayıları ve sınıf dağılımı

Aşağıdaki tablo her split için **görüntü sayısını** ve her sınıfın kaç
**görüntüde** geçtiğini gösterir.

Hatırlatma: satır toplamı > görüntü sayısı olabilir (çok etiketli görüntüler).

In [ ]:
import data_inspection

counts = split_result["counts"]
distribution = split_result["class_distribution"]

print("=" * 66)
print("GÖRÜNTÜ SAYILARI")
print("=" * 66)
total_rows = 0
for split_name in split_dataset.SPLIT_NAMES:
    label = {"train": "Train", "validation": "Validation", "test": "Test"}[split_name]
    print(f"{label:<12}: {counts[split_name]:>5}")
    total_rows += counts[split_name]
print("-" * 66)
print(f"{'TOPLAM':<12}: {total_rows:>5}")
print()

print("=" * 78)
print("SINIF BAŞINA GÖRÜNTÜ SAYISI (çok etiketli olabilir)")
print("=" * 78)
header = f"{'Sınıf':<18}{'Train':>10}{'Validation':>13}{'Test':>10}{'Toplam':>10}"
print(header)
print("-" * 78)
for class_name in data_inspection.EXPECTED_CLASSES:
    train_count = distribution["train"].get(class_name, 0)
    validation_count = distribution["validation"].get(class_name, 0)
    test_count = distribution["test"].get(class_name, 0)
    print(
        f"{class_name:<18}{train_count:>10}{validation_count:>13}"
        f"{test_count:>10}{train_count + validation_count + test_count:>10}"
    )
print("-" * 78)
for split_name, label in (("train", "Train"), ("validation", "Validation"), ("test", "Test")):
    print(
        f"{label + ' sütun toplamı':<18}"
        f"{sum(distribution[split_name].values()):>10}  "
        f"(görüntü sayısı: {counts[split_name]})"
    )
print()
print("Not: Sütun toplamı görüntü sayısından büyükse nedeni çok etiketli görüntülerdir.")

## B4) Kesişim kontrolü

Bu kontrol, aynı görüntünün iki grupta yer almadığını doğrular.
Kesişim sıfırdan büyükse ayrım hatalıdır ve `create_split` zaten hata verir.
Burada ayrıca ekrana yazdırıyoruz.

In [ ]:
overlap = split_result["overlap"]

print("=" * 60)
print("KESİŞİM KONTROLÜ")
print("=" * 60)
for pair_name, value in overlap["overlaps"].items():
    status = "TEMİZ" if value == 0 else "HATA"
    print(f"{pair_name:<24}: {value} görüntü  [{status}]")
print()
print("Toplam benzersiz görüntü :", overlap["total_unique_images"])
print("Toplam CSV satırı        :", overlap["total_rows"])
print("Kesişim var mı           :", overlap["has_overlap"])

if overlap["has_overlap"]:
    raise RuntimeError("Split grupları arasında kesişim var! Ayrım güvenilir değil.")
print()
print("Sonuç: üç grup tamamen ayrık, veri sızıntısı yok.")

## B5) Otomatik testi çalıştırma

`tests/test_split.py` yalnızca bir şeyi kontrol eder: kesişim sıfır mı?
CSV oluştuğu için test doğrudan gerçek `splits/neu_det_split.csv` dosyasını okur.

In [ ]:
import subprocess
import sys

test_file = Path(REPO_DIR) / "tests" / "test_split.py"
if not test_file.is_file():
    raise RuntimeError(f"Test dosyası bulunamadı: {test_file}")

completed = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "-v", "--no-header"],
    capture_output=True,
    text=True,
)
print(completed.stdout.strip())
if completed.returncode != 0:
    print(completed.stderr.strip())
    raise RuntimeError(f"Test başarısız (çıkış kodu {completed.returncode}).")
print("Test geçti.")

## B6) CSV'yi Google Drive'a kaydetme

Ayrım dosyası hem repo içindeki `splits/` klasöründe hem Drive'daki `outputs/`
klasöründe tutulur. Böylece Colab oturumu kapansa da dosya kaybolmaz.

Drive bağlantısı yoksa bu hücre hata verir; kayıt başarılı gibi gösterilmez.

In [ ]:
import shutil
from pathlib import Path

if not globals().get("DRIVE_READY", False):
    raise RuntimeError(
        "Google Drive bağlantısı doğrulanmadı. "
        "C bölümündeki mount hücresini çalıştırıp bu hücreyi tekrar deneyin."
    )

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/celik-kusur-tespiti/outputs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

drive_csv_path = DRIVE_OUTPUT_DIR / SPLIT_CSV_PATH.name
shutil.copy2(SPLIT_CSV_PATH, drive_csv_path)

if not drive_csv_path.is_file() or drive_csv_path.stat().st_size == 0:
    raise RuntimeError(f"CSV Drive'a kaydedilemedi: {drive_csv_path}")

repo_size = SPLIT_CSV_PATH.stat().st_size
drive_size = drive_csv_path.stat().st_size
if repo_size != drive_size:
    raise RuntimeError(
        f"Dosya boyutları uyuşmuyor (repo={repo_size}, drive={drive_size}). "
        "Kopyalama eksik olabilir."
    )

print("Repo  :", SPLIT_CSV_PATH, f"({repo_size / 1024:.1f} KB)")
print("Drive :", drive_csv_path, f"({drive_size / 1024:.1f} KB)")
print()
print("CSV satır sayısı (başlık hariç):", overlap["total_rows"])
print()
print("Test seti yalnızca nihai değerlendirme içindir; eğitimde ve")
print("parametre seçiminde kullanılmayacaktır.")

---
## Aşama 1B tamamlandı

Bu noktada:
- Her görüntü tek satırda, tüm sınıflarıyla CSV'de.
- Train / Validation / Test grupları tamamen ayrık.
- Ayrım `seed=42` ile tekrarlanabilir.

**Sıradaki adım:** Aşama 1C – XML → YOLO etiket dönüşümü ve `data.yaml` hazırlığı.

> Hatırlatma: Test seti şimdi kilitlenmiştir. Sonraki aşamalarda model seçimi,
> hiperparametre ayarı ve erken durdurma **yalnızca** train + validation ile
> yapılacak; test seti sadece en sonda bir kez ölçülecektir.

---
# Aşama 2 – XML → YOLO Dönüşümü

Bu bölüm Pascal VOC XML etiketlerini **Ultralytics YOLO** formatına çevirir.

## YOLO satır formatı
```
class_id x_center y_center width height
```
Tüm koordinatlar **0–1 arasında normalize** edilir.

## Sabit sınıf sıralaması
| ID | Sınıf |
|---|---|
| 0 | crazing |
| 1 | inclusion |
| 2 | patches |
| 3 | pitted_surface |
| 4 | rolled-in_scale |
| 5 | scratches |

Bu sıralama **değiştirilemez**. Model ağırlıkları bu ID'lere bağlı olacak.

## Kapsam dışı
Model eğitimi, model indirme, augmentation, yeni görüntü ekleme yapılmaz.
Bu aşama yalnızca veri hazırlığı ve görsel doğrulamadır.

## C1) Modülü hazırlama ve split CSV'sini bulma

`src/prepare_data.py`, Aşama 1A'daki XML okuma fonksiyonunu
(`data_inspection.parse_voc_annotation`) yeniden kullanır. XML okuma kodu
ikinci kez yazılmadı.

Split CSV'si normalde repo içindeki `splits/neu_det_split.csv` dosyasıdır.
Repo'da yoksa (örneğin temiz bir klonda henüz Aşama 1B çalıştırılmadıysa)
Drive'daki kopya kullanılır.

In [ ]:
import importlib
import shutil
import sys
from pathlib import Path

if "REPO_DIR" not in globals():
    raise RuntimeError(
        "REPO_DIR tanımlı değil. Lütfen önce Aşama 1A hücrelerini (B bölümü) çalıştırın."
    )

SRC_DIR = Path(REPO_DIR) / "src"
if not (SRC_DIR / "prepare_data.py").is_file():
    raise RuntimeError(
        f"prepare_data.py bulunamadı: {SRC_DIR / 'prepare_data.py'}\n"
        "Dosyalar GitHub'a push edilmiş mi kontrol edin ve repo'yu güncelleyin."
    )
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import prepare_data
except Exception as error:
    raise RuntimeError(f"prepare_data modülü import edilemedi: {error}") from error

importlib.reload(prepare_data)

# --- Split CSV'sini belirle: repo -> (yoksa) Drive ---
REPO_SPLIT_CSV = Path(REPO_DIR) / "splits" / "neu_det_split.csv"
DRIVE_SPLIT_CSV = Path("/content/drive/MyDrive/celik-kusur-tespiti/outputs/neu_det_split.csv")

if REPO_SPLIT_CSV.is_file():
    SPLIT_CSV_PATH = REPO_SPLIT_CSV
    print("Split CSV (repo):", SPLIT_CSV_PATH)
elif DRIVE_SPLIT_CSV.is_file():
    SPLIT_CSV_PATH = DRIVE_SPLIT_CSV
    # Drive'daki dosyayı repo'ya da kopyalıyoruz ki sonraki hücreler tutarlı olsun.
    REPO_SPLIT_CSV.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_SPLIT_CSV, REPO_SPLIT_CSV)
    SPLIT_CSV_PATH = REPO_SPLIT_CSV
    print("Split CSV Drive'dan alındı ve repo'ya kopyalandı:", SPLIT_CSV_PATH)
else:
    raise RuntimeError(
        "Split CSV bulunamadı.\n"
        f"  Denenen repo yolu : {REPO_SPLIT_CSV}\n"
        f"  Denenen Drive yolu: {DRIVE_SPLIT_CSV}\n"
        "Lütfen önce Aşama 1B hücrelerini (B2) çalıştırın."
    )

print("Sınıf eşleştirmesi:", prepare_data.CLASS_TO_ID)
print("Beklenen toplam kutu (Aşama 1A):", prepare_data.EXPECTED_TOTAL_BOXES)

## C2) Dönüşümü çalıştırma

Bu hücre şunları yapar:
- CSV'yi okur ve doğrular (eksik sütun, yinelenen görüntü, bilinmeyen split → hata).
- `data/processed/neu_det_yolo/` altında `images/{train,val,test}` ve
  `labels/{train,val,test}` klasörlerini oluşturur.
- Görüntüleri **orijinal haliyle** kopyalar (yeniden boyutlandırma yok).
- Her görüntü için aynı isimli `.txt` etiketi yazar.
- `data.yaml` dosyasını üretir.

Hedef dosya zaten varsa hata verir (sessizce üzerine yazmaz).
Bu yüzden hücreyi ikinci kez çalıştırırsanız önce çıktı klasörünü silmeniz gerekir —
bir sonraki hücre bunu sizin onayınızla yapar.

In [ ]:
from pathlib import Path

YOLO_OUTPUT_DIR = Path(REPO_DIR) / "data" / "processed" / "neu_det_yolo"
YOLO_YAML_PATH = YOLO_OUTPUT_DIR / "data.yaml"

# Önceki bir çalıştırmadan kalan çıktı varsa üzerine sessizce yazmıyoruz.
if YOLO_OUTPUT_DIR.exists() and any(YOLO_OUTPUT_DIR.rglob("*")):
    print("UYARI: çıktı klasörü zaten dolu:", YOLO_OUTPUT_DIR)
    print("Temizleyip yeniden üretmek için bir sonraki hücreyi kullanın.")
    print("Temizlemeden çalıştırmak FileExistsError verir.")
else:
    print("Çıktı klasörü hazır:", YOLO_OUTPUT_DIR)

### Çıktı klasörünü temizleme (isteğe bağlı)

Önceki çalıştırmanın çıktısını silmek için bu hücreyi **bilinçli olarak**
çalıştırın. Sadece `data/processed/neu_det_yolo` klasörünü siler; başka hiçbir
yere dokunmaz.

In [ ]:
# Bu hücreyi yalnızca yeniden üretmek istediğinizde çalıştırın.
RESET_OUTPUT = False  # True yaparsanız klasör silinir.

if RESET_OUTPUT and YOLO_OUTPUT_DIR.exists():
    shutil.rmtree(YOLO_OUTPUT_DIR)
    print("Silindi:", YOLO_OUTPUT_DIR)
elif RESET_OUTPUT:
    print("Silinecek klasör yok:", YOLO_OUTPUT_DIR)
else:
    print("Temizleme atlandı (RESET_OUTPUT = False).")

In [ ]:
conversion = prepare_data.prepare_yolo_dataset(
    dataset_root=EXTRACT_DIR,
    split_csv=SPLIT_CSV_PATH,
    output_dir=YOLO_OUTPUT_DIR,
    yaml_path=YOLO_YAML_PATH,
)

print("=" * 64)
print("GÖRÜNTÜ SAYILARI")
print("=" * 64)
for key, label in (("train", "Train"), ("validation", "Validation"), ("test", "Test")):
    print(f"{label:<12}: {conversion['split_image_counts'][key]:>5}")
print("-" * 64)
print(f"{'TOPLAM':<12}: {conversion['total_images']:>5}")
print()

print("=" * 64)
print("BOUNDING BOX SAYILARI")
print("=" * 64)
for key, label in (("train", "Train"), ("validation", "Validation"), ("test", "Test")):
    print(f"{label:<12}: {conversion['split_box_counts'][key]:>5}")
print("-" * 64)
print(f"{'TOPLAM':<12}: {conversion['total_boxes']:>5}")
print(f"{'BEKLENEN':<12}: {conversion['expected_total_boxes']:>5}")
print("Eşleşiyor mu :", conversion["boxes_match_expected"])

## C3) Kutu sayısı kontrolü (kritik)

Aşama 1A incelemesinde toplam **4189** bounding box ölçüldü.
Dönüşüm sonucu bu sayıdan farklıysa bir şeyler yanlış gitmiş demektir.

Bu hücre farkı **hata olarak** bildirir ve devam etmez; böylece eksik veya
fazla kutu ile eğitime geçilmez.

In [ ]:
if not conversion["boxes_match_expected"]:
    difference = conversion["total_boxes"] - conversion["expected_total_boxes"]
    print("=" * 64)
    print("HATA: KUTU SAYISI BEKLENENLE UYUŞMUYOR")
    print("=" * 64)
    print("Beklenen :", conversion["expected_total_boxes"])
    print("Bulunan  :", conversion["total_boxes"])
    print("Fark     :", difference, "(pozitif = fazla, negatif = eksik)")
    print()
    print("Olası nedenler:")
    print(" 1. Aşama 1A farklı bir veri seti sürümüyle çalıştırıldı.")
    print(" 2. Bazı XML'lerde kutu sayısı değişmiş / dosyalar değiştirilmiş.")
    print(" 3. Split CSV'si farklı bir görüntü kümesinden üretilmiş.")
    print(" 4. Eksik çıkarılmış ZIP (bazı görüntüler yok).")
    print()
    print("Sınıf bazında kutu sayıları:")
    for class_name, count in conversion["class_box_counts"].items():
        print(f"  {class_name:<18}: {count:>5}")
    raise RuntimeError(
        f"Kutu sayısı beklenen {conversion['expected_total_boxes']} değil, "
        f"{conversion['total_boxes']} bulundu. Dönüşüm tamamlanmış sayılmaz."
    )

print("Kutu sayısı beklenen değerle birebir eşleşiyor:", conversion["total_boxes"])

## C4) Sınıf dağılımı ve data.yaml

Sınıf başına **görüntü** ve **kutu** sayıları ayrı gösterilir: bir görüntü
birden fazla sınıf içerebilir, bu yüzden görüntü sayıları toplamı görüntü
sayısından büyük olabilir.

In [ ]:
print("=" * 72)
print("SINIF DAĞILIMI")
print("=" * 72)
print(f"{'ID':<4}{'Sınıf':<20}{'Görüntü':>10}{'Kutu':>10}")
print("-" * 72)
for class_id, class_name in enumerate(prepare_data.CLASS_NAMES):
    print(
        f"{class_id:<4}{class_name:<20}"
        f"{conversion['class_image_counts'][class_name]:>10}"
        f"{conversion['class_box_counts'][class_name]:>10}"
    )
print("-" * 72)
print(f"{'':<4}{'Çok sınıflı görüntü':<20}{conversion['multi_class_images']:>10}")
print()
print("=" * 72)
print("DATA.YAML")
print("=" * 72)
print(YOLO_YAML_PATH.read_text(encoding="utf-8"))

## C5) Otomatik test

`tests/test_xml_to_yolo.py` sentetik bir XML ile şunları doğrular:
- XML koordinatlarının okunması
- sınıf ID eşleştirmesi
- normalize YOLO koordinatlarının doğru hesaplanması

In [ ]:
import subprocess
import sys

test_file = Path(REPO_DIR) / "tests" / "test_xml_to_yolo.py"
if not test_file.is_file():
    raise RuntimeError(f"Test dosyası bulunamadı: {test_file}")

completed = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "-v", "--no-header"],
    capture_output=True,
    text=True,
)
print(completed.stdout.strip())
if completed.returncode != 0:
    print(completed.stderr.strip())
    raise RuntimeError(f"Test başarısız (çıkış kodu {completed.returncode}).")
print("Test geçti.")

## C6) Görsel doğrulama

6–8 gerçek görüntü seçilir. Kutular **XML'den değil, yazdığımız TXT
dosyalarından** okunup piksel koordinatlarına geri çevrilerek çizilir.

Bu sayede TXT dosyasının gerçekten doğru olduğu görülür. Ayrıca her kutu için
XML koordinatlarıyla farkı ölçülür; fark 0.01 pikselden büyükse hata verir.

In [ ]:
YOLO_SAMPLES_PNG = Path(REPO_DIR) / "results" / "yolo_conversion_samples.png"

visual_check = prepare_data.plot_yolo_samples(
    dataset_root=EXTRACT_DIR,
    split_csv=SPLIT_CSV_PATH,
    output_dir=YOLO_OUTPUT_DIR,
    output_png=YOLO_SAMPLES_PNG,
    max_samples=8,
    seed=42,
)

print("Görsel kaydedildi :", visual_check["output_png"])
print("Örnek sayısı      :", visual_check["sample_count"])
print("Karşılaştırılan kutu:", len(visual_check["comparisons"]))
print("Maks. sapma (px)  :", f"{visual_check['max_deviation_px']:.10f}")
print()

if visual_check["max_deviation_px"] > 0.01:
    raise RuntimeError(
        f"YOLO kutuları XML kutularıyla uyuşmuyor. "
        f"Maksimum sapma {visual_check['max_deviation_px']} piksel."
    )
print("Sonuç: TXT'den geri çizilen kutular XML kutularıyla aynı konumda.")
print()
print("Seçilen görüntüler:")
for path in visual_check["sample_images"]:
    print(" -", path)

from IPython.display import Image as NotebookImage, display
display(NotebookImage(filename=visual_check["output_png"]))

## C7) Google Drive'a kalıcı kopya

Hazırlanan YOLO veri seti Drive'a kopyalanır ve dosya sayıları karşılaştırılır.
Drive bağlantısı yoksa hücre hata verir.

In [ ]:
from pathlib import Path

if not globals().get("DRIVE_READY", False):
    raise RuntimeError(
        "Google Drive bağlantısı doğrulanmadı. "
        "C bölümündeki mount hücresini çalıştırıp bu hücreyi tekrar deneyin."
    )

DRIVE_YOLO_DIR = Path("/content/drive/MyDrive/celik-kusur-tespiti/processed/neu_det_yolo")

drive_result = prepare_data.mirror_to_drive(YOLO_OUTPUT_DIR, DRIVE_YOLO_DIR)

print("Kaynak dosya sayısı :", drive_result["source_files"])
print("Drive dosya sayısı  :", drive_result["drive_files"])
print("Kopyalanan          :", drive_result["copied"])
print("Zaten vardı (atlandı):", drive_result["skipped"])
print("Doğrulama           :", "BAŞARILI" if drive_result["verified"] else "BAŞARISIZ")
print()

if not drive_result["verified"]:
    raise RuntimeError(
        f"Drive kopyası eksik: kaynak {drive_result['source_files']} dosya, "
        f"hedef {drive_result['drive_files']} dosya."
    )

print("Yerel :", YOLO_OUTPUT_DIR)
print("Drive :", drive_result["drive_dir"])
print("YAML  :", YOLO_YAML_PATH)

---
## Aşama 2 tamamlandı

Bu noktada:
- 1800 görüntü YOLO klasör yapısında (1260 train / 270 val / 270 test).
- Her görüntü için normalize koordinatlı bir `.txt` etiketi var.
- Sınıf ID sıralaması sabit ve `data.yaml` içinde tanımlı.
- Kutular görsel olarak doğrulandı (TXT'den geri çizim).

**Sıradaki adım:** Aşama 3 – Colab'da YOLO model eğitimi.

> Test seti kuralı: `images/test` klasörü eğitimde ve hiperparametre seçiminde
> **kullanılmayacak**. Yalnızca en sonda bir kez değerlendirilecek.

---
# Aşama 3 – YOLO Model Eğitimi

Bu bölüm `yolo26n.pt` modelini NEU-DET üzerinde **bir kez** eğitir.

## En önemli kural: TEST SETİ KULLANILMAZ

Eğitim, `test:` anahtarı **içermeyen** ayrı bir YAML dosyasıyla yapılır.
Yani Ultralytics eğitim sırasında test görüntülerine erişemez:

| Kullanılan | Kullanılmayan |
|---|---|
| `images/train` (1260) | `images/test` (270) |
| `images/val` (270) | test etiketleri |

Test seti yalnızca ilerideki **final değerlendirme** aşamasında, bir kez
kullanılacak. Bu aşamada test üzerinde tahmin, değerlendirme veya confusion
matrix analizi **yapılmaz**.

## Deney sınırı
Toplam en fazla 3 eğitim denemesi yapılabilir. Bu bölüm yalnızca
**1. denemeyi** (`yolo26n_exp01`) çalıştırır. Sonuç incelenmeden ikinci deneme
başlatılmaz.

## Bu bölüm bağımsız çalışır
Önceki hücrelerin değişkenlerine ihtiyaç duymaz: repo'yu, Drive'ı ve veri
setini kendisi kontrol eder.

## D1) Ortam hazırlığı (repo + `src`)

Önceki hücreleri çalıştırmadıysanız sorun değil. Bu hücre repo'yu bulur,
yoksa GitHub'dan klonlar, gerekirse fast-forward ile günceller ve `src`
klasörünü import yoluna ekler.

**Önemli:** `git pull` yalnızca `--ff-only` ile yapılır. Yerel değişiklikler
varsa pull başarısız olur ve bu durum raporlanır; hiçbir dosya zorla
değiştirilmez.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/mertkk1623/celik-kusur-tespiti.git"
REPO_DIR = Path("/content/celik-kusur-tespiti")

# 1) Repo var mı? Yoksa klonla.
if (REPO_DIR / ".git").is_dir():
    print("Repo mevcut, klonlanmıyor:", REPO_DIR)
else:
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} var ama bir git deposu değil. "
            "Klasörü silip hücreyi yeniden çalıştırın."
        )
    print("Repo klonlanıyor...")
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if clone.returncode != 0:
        raise RuntimeError(
            "Repository klonlanamadı:\n" + clone.stderr.strip()
        )
    print("Klonlandı:", REPO_DIR)

# 2) Fast-forward ile güncelle (zorla değişiklik yok).
pull = subprocess.run(
    ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
    capture_output=True, text=True,
)
if pull.returncode == 0:
    print("Güncelleme:", (pull.stdout.strip() or "zaten güncel"))
else:
    print("UYARI: 'git pull --ff-only' başarısız oldu, mevcut kod kullanılacak.")
    print("Sebep:", pull.stderr.strip())

# 3) src'yi import yoluna ekle ve modülü yükle.
SRC_DIR = REPO_DIR / "src"
if not (SRC_DIR / "train.py").is_file():
    raise RuntimeError(
        f"train.py bulunamadı: {SRC_DIR / 'train.py'}\n"
        "Dosyalar GitHub'a push edilmiş mi kontrol edin."
    )
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import train
importlib.reload(train)

print("Modül hazır :", train.__file__)
print("Model       :", train.MODEL_NAME)
print("Deneme adı  :", train.EXPERIMENT_NAME)

## D2) Google Drive bağlantısı

Model ve eğitim çıktıları Drive'da kalıcı olacak. Colab oturumu kapandığında
`/content` silinir, Drive kalır.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError(
        f"Google Drive bağlanamadı: {DRIVE_ROOT}\n"
        "Önceki mount hücresini çalıştırıp izin verdiğinizden emin olun."
    )
DRIVE_READY = True
print("Drive bağlantısı doğrulandı:", DRIVE_ROOT)

## D3) Ortam ve GPU kontrolü

Beklenen GPU: **NVIDIA Tesla T4** (Colab Pro ücretsiz/basic GPU).

CUDA yoksa eğitim **başlatılmaz**. CPU'da 50 epoch pratikte bitmez; başlatmak
sadece zaman kaybı olur.

Ultralytics kurulu değilse yalnızca o kurulur. PyTorch ve CUDA paketlerine
dokunulmaz.

In [ ]:
env_info = train.check_environment(require_cuda=True)

print("=" * 64)
print("ORTAM BİLGİLERİ")
print("=" * 64)
print("Python        :", env_info["python_version"])
print("PyTorch       :", env_info["torch_version"])
print("CUDA derlemesi:", env_info["torch_cuda_version"])
print("Ultralytics   :", env_info["ultralytics_version"])
print("CUDA var mı   :", env_info["cuda_available"])
print("GPU modeli    :", env_info["gpu_name"])
print("VRAM          :", env_info["vram_gb"], "GB")

if "T4" in str(env_info["gpu_name"]):
    print()
    print("GPU beklendiği gibi: NVIDIA Tesla T4")
else:
    print()
    print("NOT: Beklenen GPU Tesla T4 değil. Eğitim devam edebilir,")
    print("ancak süre ve batch boyutu farklı olabilir.")

In [ ]:
install_info = train.ensure_ultralytics_installed()

print("=" * 64)
print("ULTRALYTICS")
print("=" * 64)
print("Bu çalıştırmada kuruldu mu:", install_info["installed_now"])
print("Sürüm                       :", install_info["version"])
print()
print("PyTorch veya CUDA paketleri güncellenmedi.")

# Kurulum sonrası sürüm bilgisini tazele.
env_info = train.check_environment(require_cuda=True)
print("Doğrulama sonrası Ultralytics:", env_info["ultralytics_version"])

## D4) Veri seti hazırlığı ve doğrulaması

Sırasıyla:
1. Drive'daki hazır veri seti yerelde yoksa `/content` altına kopyalanır.
   Yerelde zaten varsa **üzerine yazılmaz, yeniden oluşturulmaz**.
2. `data.yaml` var mı?
3. `images/train` tam 1260 görüntü mü?
4. `images/val` tam 270 görüntü mü?
5. Her görüntünün TXT etiketi var mı?

Bunlardan biri eksikse eğitim **durdurulur**.

In [ ]:
DRIVE_DATASET_DIR = Path("/content/drive/MyDrive/celik-kusur-tespiti/processed/neu_det_yolo")
LOCAL_DATASET_DIR = Path("/content/celik-kusur-tespiti/data/processed/neu_det_yolo")

dataset_prep = train.prepare_local_dataset(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
print(dataset_prep["message"])
print("Kullanılan klasör:", dataset_prep["dataset_dir"])
print()

dataset_check = train.verify_dataset(dataset_prep["dataset_dir"])

print("=" * 64)
print("VERİ SETİ DOĞRULAMASI")
print("=" * 64)
print("data.yaml        :", dataset_check["data_yaml"])
print("Train görüntü    :", dataset_check["train_images"], "(beklenen 1260)")
print("Validation       :", dataset_check["val_images"], "(beklenen 270)")
print("Eksik TXT (train):", dataset_check["train_missing_labels"])
print("Eksik TXT (val)  :", dataset_check["val_missing_labels"])
print("Sonuç            :", "GEÇTİ" if dataset_check["ok"] else "BAŞARISIZ")

if not dataset_check["ok"]:
    raise RuntimeError("Veri seti doğrulaması başarısız, eğitim başlatılmıyor.")

## D5) Eğitim YAML'i — test setini koruma adımı

Bu hücre `data_train_only.yaml` dosyasını üretir. İçinde `train` ve `val`
vardır, **`test` yoktur**.

`data.yaml` (test yolu içeren orijinal dosya) değiştirilmez; olduğu gibi
kalır. Eğitimde onu değil, bu yeni dosyayı kullanıyoruz.

In [ ]:
training_yaml = train.write_training_yaml(dataset_prep["dataset_dir"])

print("Eğitim YAML:", training_yaml["yaml_path"])
print()
print(training_yaml["content"])
print("Doğrulama: bu dosyada 'test:' anahtarı yok.")
print("Test seti eğitimde kullanılmayacak.")

## D6) Eğitim ayarları (çalıştırmadan önce gözden geçirin)

Aşağıdaki hücre **eğitimi başlatmaz**, sadece ne yapılacağını yazdırır.
Kontrol ettikten sonra bir sonraki hücreyi çalıştırın.

| Ayar | Değer |
|---|---|
| Model | `yolo26n.pt` |
| epochs | 50 |
| imgsz | 640 |
| batch | -1 (GPU belleğine göre otomatik) |
| patience | 15 |
| device | 0 |
| workers | 2 |
| seed | 42 |
| deterministic | True |
| pretrained | True |
| amp | True |
| plots / save / val | True |

Optimizer ve augmentation ayarlarına dokunulmadı — Ultralytics varsayılanları
kullanılıyor.

In [ ]:
DRIVE_TRAINING_DIR = Path("/content/drive/MyDrive/celik-kusur-tespiti/training")

print("=" * 64)
print("EĞİTİM AYARLARI (henüz başlamadı)")
print("=" * 64)
print("Model         :", train.MODEL_NAME)
print("Deneme adı    :", train.EXPERIMENT_NAME)
print("Veri (YAML)   :", training_yaml["yaml_path"])
print("Çıktı klasörü :", DRIVE_TRAINING_DIR / train.EXPERIMENT_NAME)
print()
for key, value in train.TRAIN_SETTINGS.items():
    print(f"  {key:<14}: {value}")
print()
print("Toplam deney sınırı: 3 | Bu çalıştırma: 1. deneme")

## D7) Eğitimi başlatma ⚠️

**Bu hücre gerçek eğitimi başlatır.** T4 GPU'da yaklaşık 10–25 dakika sürer
(veri setine ve batch boyutuna göre değişir).

Notlar:
- Eğitim çıktıları doğrudan Drive'a yazılır (Colab kopmasına karşı).
- `yolo26n_exp01` klasörü zaten doluysa hücre **hata verir**, üzerine yazmaz.
- Eğitim otomatik olarak tekrarlanmaz. Hata olursa durur ve size bildirir.

In [ ]:
training_run = train.train_yolo(
    data_yaml=training_yaml["yaml_path"],
    project_dir=DRIVE_TRAINING_DIR,
    run_name=train.EXPERIMENT_NAME,
    model_name=train.MODEL_NAME,
)

print()
print("=" * 64)
print("EĞİTİM TAMAMLANDI")
print("=" * 64)
print("Çıktı klasörü:", training_run["run_dir"])
print("Kullanılan ayarlar:")
for key, value in training_run["settings"].items():
    print(f"  {key:<14}: {value}")

## D8) Validation sonuçları

Metrikler **`results.csv` dosyasından** okunur, tahmin edilmez.

En iyi epoch, Ultralytics'in `best.pt` seçiminde kullandığı fitness değerine
göre belirlenir:
```
fitness = 0.1 * mAP50 + 0.9 * mAP50-95
```

In [ ]:
training_results = train.read_training_results(training_run["run_dir"])

print("=" * 64)
print("EĞİTİM SONUÇLARI")
print("=" * 64)
print("Model                :", train.MODEL_NAME)
print("Ultralytics sürümü   :", env_info["ultralytics_version"])
print("Tamamlanan epoch     :", training_results["epochs_completed"])
print("Eğitim süresi        :", training_results["duration_minutes"], "dakika")
print("GPU                  :", env_info["gpu_name"], f"({env_info['vram_gb']} GB)")
print()

best_metrics = training_results["best_epoch_metrics"]
print("=" * 64)
print("EN İYİ EPOCH:", training_results["best_epoch"])
print("=" * 64)
print("Precision  :", best_metrics["precision"])
print("Recall     :", best_metrics["recall"])
print("mAP50      :", best_metrics["mAP50"])
print("mAP50-95   :", best_metrics["mAP50_95"])
print("Fitness    :", training_results["best_fitness"])
print()

final_metrics = training_results["final_epoch_metrics"]
print("Son epoch:", final_metrics["epoch"])
print("  Precision:", final_metrics["precision"], "| Recall:", final_metrics["recall"])
print("  mAP50    :", final_metrics["mAP50"], "| mAP50-95:", final_metrics["mAP50_95"])
print()
print("best.pt :", training_results["best_pt"])
print("last.pt :", training_results["last_pt"])

## D9) Eğitim grafikleri

Ultralytics'in ürettiği grafikler Drive'daki eğitim klasöründe korunuyor.
Aşağıda bulunanların yolları listelenir ve ilk ikisi gösterilir.

In [ ]:
from IPython.display import Image as NotebookImage, display

print("Bulunan grafikler:")
for name, path in training_results["plot_files"].items():
    print(f"  {name:<32} {path}")

if not training_results["plot_files"]:
    print("  (grafik dosyası bulunamadı)")

for name in ("results.png", "confusion_matrix.png"):
    path = training_results["plot_files"].get(name)
    if path:
        print()
        print(name)
        display(NotebookImage(filename=path))

## D10) best.pt dosyasını Drive'a kaydetme

`weights/best.pt` → `/content/drive/MyDrive/celik-kusur-tespiti/models/best.pt`

Hedef dosya zaten varsa **üzerine yazılmaz**; durum raporlanır.
`last.pt` ve `results.csv` zaten eğitim klasöründe Drive'da duruyor.

In [ ]:
DRIVE_BEST_PT = Path("/content/drive/MyDrive/celik-kusur-tespiti/models/best.pt")

if training_results["best_pt"] is None:
    raise RuntimeError(
        "best.pt bulunamadı. Eğitim tamamlanmamış olabilir; "
        "results.csv ve logları kontrol edin."
    )

save_result = train.save_best_model(training_results["best_pt"], DRIVE_BEST_PT)

print("=" * 64)
print("MODEL KAYDI")
print("=" * 64)
print("Durum          :", save_result["status"])
print("Hedef          :", save_result["destination"])
print("Kaynak boyut   :", save_result["source_size_bytes"], "bayt")
print("Hedef boyut    :", save_result["destination_size_bytes"], "bayt")
print()
print(save_result["message"])

## D11) Özeti JSON olarak kaydetme

Tüm eğitim bilgisi tek dosyada toplanır; raporlama ve tekrar üretilebilirlik
için yararlıdır.

In [ ]:
from datetime import datetime, timezone

summary = {
    "stage": "3",
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model_name": train.MODEL_NAME,
    "experiment_name": train.EXPERIMENT_NAME,
    "ultralytics_version": env_info["ultralytics_version"],
    "torch_version": env_info["torch_version"],
    "python_version": env_info["python_version"],
    "gpu_name": env_info["gpu_name"],
    "vram_gb": env_info["vram_gb"],
    "settings": training_run["settings"],
    "data_yaml_used": training_yaml["yaml_path"],
    "test_set_used_in_training": False,
    "dataset_check": dataset_check,
    "dataset_prep": dataset_prep,
    "results": {
        key: value for key, value in training_results.items()
        if key not in ("plot_files",)
    },
    "plot_files": training_results["plot_files"],
    "best_pt_save": save_result,
}

summary_path = train.write_training_summary(
    summary,
    Path(training_run["run_dir"]) / "training_summary.json",
)

print("Özet yazıldı:", summary_path)
print("Boyut        :", summary_path.stat().st_size, "bayt")

---
## Aşama 3 tamamlandı (1. deneme)

- Model `yolo26n.pt` ile **yalnızca train + val** üzerinde eğitildi.
- Test seti bu aşamada okunmadı ve değerlendirilmedi.
- `best.pt` ve `last.pt` Drive'da korunuyor.
- `results.csv` ve tüm grafikler eğitim klasöründe.

**Sıradaki adım:** Sonuçlar incelenecek. Gerekirse 2. ve 3. deneme
(deney sınırı: toplam 3) yapılacak. Model seçimi **yalnızca validation**
sonuçlarına göre verilecek.

**Test seti** en sonda, ayrı bir final değerlendirme aşamasında bir kez
kullanılacak.